# ETAPA 11.1 - Agente completo com Qwen3-8B + QLoRA
Este notebook inicia o gerador oficial em uma **Tesla T4**, valida o adapter da ETAPA 5 por SHA-256 e expõe uma URL HTTPS temporária protegida por token. O Streamlit local continua responsável por SQLite, RAG, LangChain, LangGraph, safety e auditoria. Não execute na GTX 1650.

In [ ]:
!nvidia-smi
import platform
print(platform.python_version())
assert platform.python_version_tuple()[:2] == ('3', '12'), 'Selecione um runtime Colab com Python 3.12'

In [ ]:
import os, subprocess
REPOSITORY = 'https://github.com/mo1sess/tech-challenge-fase3.git'
PROJECT = '/content/tech-challenge-fase3'
if not os.path.exists(PROJECT):
    subprocess.run(['git', 'clone', REPOSITORY, PROJECT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT, 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
print(os.getcwd())

In [ ]:
%pip install -q -e . -r requirements/full-agent-gpu.txt
!python scripts/validate_full_agent.py

## Adapter oficial
Coloque `qwen3_8b_qlora_stage5_evidence.zip` no seu Google Drive. Altere somente o caminho abaixo. O arquivo não deve ser publicado como código-fonte.

In [ ]:
from google.colab import drive
from pathlib import Path
import tempfile
drive.mount('/content/drive')
EVIDENCE_ZIP = Path('/content/drive/MyDrive/qwen3_8b_qlora_stage5_evidence.zip')
assert EVIDENCE_ZIP.is_file(), f'Arquivo não encontrado: {EVIDENCE_ZIP}'
from clinical_assistant.acquisition.common import safe_extract_zip
evidence_root = Path(tempfile.mkdtemp(prefix='stage5-evidence-', dir='/content'))
safe_extract_zip(EVIDENCE_ZIP, evidence_root)
adapters = [p.parent for p in evidence_root.rglob('adapter_model.safetensors') if p.parent.name == 'adapter']
assert len(adapters) == 1, f'Esperado um adapter final; encontrados: {adapters}'
ADAPTER = adapters[0]
print('Adapter:', ADAPTER)

In [ ]:
import secrets, time, requests
TOKEN = secrets.token_urlsafe(32)
server_env = os.environ.copy()
server_env['TECHCARE_REMOTE_TOKEN'] = TOKEN
SERVER = subprocess.Popen(
    ['python', 'scripts/serve_qwen_agent.py', '--adapter', str(ADAPTER)],
    env=server_env,
)
for _ in range(180):
    try:
        health = requests.get('http://127.0.0.1:8000/health', headers={'Authorization': f'Bearer {TOKEN}'}, timeout=3)
        if health.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('O serviço Qwen não iniciou dentro do limite esperado')
print(health.json())

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import re, threading, queue
lines = queue.Queue()
TUNNEL = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
threading.Thread(target=lambda: [lines.put(line) for line in TUNNEL.stdout], daemon=True).start()
PUBLIC_URL = None
deadline = time.time() + 90
while time.time() < deadline and not PUBLIC_URL:
    try:
        line = lines.get(timeout=2)
    except queue.Empty:
        continue
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        PUBLIC_URL = match.group(0)
assert PUBLIC_URL, 'Não foi possível obter a URL HTTPS temporária'
print('TECHCARE_REMOTE_URL=', PUBLIC_URL)
print('TECHCARE_REMOTE_TOKEN=', TOKEN)

## Conectar o Streamlit local
No PowerShell do computador local, mantenha este notebook executando e defina as três variáveis abaixo com os valores impressos. Depois inicie o Streamlit:
```powershell
$env:TECHCARE_EXECUTION_MODE = "qwen_remote"
$env:TECHCARE_REMOTE_URL = "URL_HTTPS_IMPRESSA"
$env:TECHCARE_REMOTE_TOKEN = "TOKEN_IMPRESSO"
& .\.venv\Scripts\python.exe -m streamlit run app\streamlit_app.py
```
Para gerar a evidência oficial, execute em outro PowerShell, com as mesmas variáveis: `& .\.venv\Scripts\python.exe scripts\run_remote_agent_validation.py`.